## 1. Environment Setup

In [ ]:
# Detect environment and setup paths
import os
import sys
from pathlib import Path

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules
print(f"Running in Colab: {IN_COLAB}")

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    # Set repo path in Drive
    REPO_ROOT = Path('/content/drive/MyDrive/Thesis/Ego4d-LiteSTA')
else:
    # Local development - find repo root
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / 'local_extraction').exists() and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent

LOCAL_EXTRACTION = REPO_ROOT / 'local_extraction'
print(f"Repo root: {REPO_ROOT}")
print(f"Local extraction: {LOCAL_EXTRACTION}")

# Add to Python path (order matters)
paths_to_add = [
    str(LOCAL_EXTRACTION),
    str(REPO_ROOT),
    str(LOCAL_EXTRACTION / 'core'),
    str(LOCAL_EXTRACTION / 'trackA'),
    str(LOCAL_EXTRACTION / 'trackA' / 'trackA_stageA'),
    str(LOCAL_EXTRACTION / 'trackA' / 'trackA_stageB'),
    str(LOCAL_EXTRACTION / 'trackB'),
    str(LOCAL_EXTRACTION / 'trackC'),
    str(LOCAL_EXTRACTION / 'tests'),
]
for p in paths_to_add:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(str(LOCAL_EXTRACTION))
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies if needed (Colab only)
if IN_COLAB:
    !pip install -q ultralytics pyyaml tqdm pillow
    print("Dependencies installed")

In [ ]:
# Import core modules
from core import (
    load_config, Config, is_colab, get_device,
    Paths, get_paths,
    RunLogger, create_run_logger,
)

# Check device
import torch
device = get_device()
print(f"Device: {device}")
print(f"Is Colab: {is_colab()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration Management

Load and customize YAML configurations for each track.

In [ ]:
# Load base configuration
base_cfg = load_config('base')
print("Base Config:")
print(f"  Version: {base_cfg.get('version')}")
print(f"  Demo mode: {base_cfg.get('demo.enabled')}")
print(f"  Device: {base_cfg.get('runtime.device')}")

# Show available configs
configs_dir = LOCAL_EXTRACTION / 'configs'
print(f"\nAvailable YAML configs:")
for cfg_file in sorted(configs_dir.glob('*.yaml')):
    print(f"  - {cfg_file.name}")

In [ ]:
# Configuration override dictionary
# Modify these settings to customize your run

CONFIG_OVERRIDES = {
    # Demo mode (fast smoke tests)
    'demo': {
        'enabled': True,  # Set to False for full training
        'max_samples': 10,
        'steps': 5,
    },
    
    # Runtime settings (auto-adjust for Colab)
    'runtime': {
        'device': 'auto',  # 'auto', 'cuda', or 'cpu'
        'num_workers': 2 if IN_COLAB else 0,  # Colab needs workers, local often doesn't
        'use_half': False,  # Set True for faster inference on GPU
    },
    
    # Training settings
    'training': {
        'epochs': 2 if base_cfg.get('demo.enabled') else 20,
        'batch_size': 4 if IN_COLAB else 8,
    },
    
    # Logging
    'logging': {
        'verbose': True,
        'save_checkpoints': True,
    },
}

print("Config overrides ready:")
for section, values in CONFIG_OVERRIDES.items():
    print(f"  {section}: {values}")

## 3. Paths Setup

In [ ]:
# Initialize paths with auto-detection
paths = get_paths()

print(f"Local extraction: {paths.local_extraction}")
print(f"Frames: {paths.extracted_frames}")
print(f"YOLO labels: {paths.yolo_labels}")
print(f"Manifests: {paths.manifests}")
print(f"Annotations: {paths.org_annotations}")

# Check if paths exist
print("\nPath status:")
for name, path in [
    ('Frames', paths.extracted_frames),
    ('Labels', paths.yolo_labels),
    ('Manifests', paths.manifests),
]:
    exists = Path(path).exists() if path else False
    status = '✓' if exists else '✗'
    print(f"  {status} {name}: {path}")

## 3.1 Run Logging Setup

Initialize the run logger to track configuration and results.

In [ ]:
# Run Logger utility functions for notebook orchestration
from datetime import datetime

def create_notebook_logger(track: str, stage: str = None):
    """Create a run logger for notebook-driven runs.
    
    Uses the core.create_run_logger factory for consistency.
    """
    # Use factory function from core module
    logger = create_run_logger(
        track=track,
        stage=stage,
        run_dir=paths.local_extraction / 'runs' / f'Track_{track[-1].upper()}',
    )
    logger.log_config(CONFIG_OVERRIDES)
    logger.log_note(f"Notebook run | Colab={IN_COLAB} | {datetime.now().isoformat()}")
    return logger

def log_run_results(logger: RunLogger, metrics: dict, artifacts: list = None):
    """Log results and finalize the run."""
    logger.log_metrics(metrics)
    if artifacts:
        logger.log_artifacts(artifacts)
    logger.log_end(success=True)
    logger.print_summary()
    return logger.save()

print("Run logger utilities ready")
print(f"Logs will be saved to: {paths.local_extraction / 'runs'}")

# Quick test (optional)
# test_logger = create_notebook_logger('trackA', 'test')
# print(f"Test logger ID: {test_logger.run_id}")

---
## 4. Track A: Object Detector

Run YOLO detection and head training.

In [ ]:
# Load Track A configuration
trackA_cfg = load_config('trackA', overrides=CONFIG_OVERRIDES)

print("Track A Configuration:")
print(f"  Mode: {trackA_cfg.get('stage_a.mode')}")
print(f"  K (candidates): {trackA_cfg.get('stage_a.k')}")
print(f"  YOLO weights: {trackA_cfg.get('stage_a.yolo.weights')}")
print(f"  YOLO conf: {trackA_cfg.get('stage_a.yolo.conf_thresh')}")
print(f"  Demo mode: {trackA_cfg.get('demo.enabled')}")

In [ ]:
# Run Track A Stage A (Detection)
RUN_STAGE_A = False  # Set to True to run

if RUN_STAGE_A:
    print("Running Track A Stage A...")
    
    # Create run logger
    logger_a = create_notebook_logger('trackA', 'stageA')
    logger_a.log_start()
    
    try:
        # Import and run
        sys.path.insert(0, str(LOCAL_EXTRACTION / 'trackA' / 'trackA_stageA'))
        import trackA_stageA
        
        # Run detection
        result = trackA_stageA.main()
        
        # Log results
        logger_a.log_metrics({'exit_code': result})
        logger_a.log_end(success=(result == 0))
        logger_a.print_summary()
        
    except Exception as e:
        logger_a.log_end(success=False, error=str(e))
        print(f"Stage A failed: {e}")
else:
    print("Stage A skipped (set RUN_STAGE_A = True to run)")

In [ ]:
# Run Track A Stage B (Cropping + Head Manifests)
RUN_STAGE_B = False  # Set to True to run

if RUN_STAGE_B:
    print("Running Track A Stage B...")
    
    # Create run logger
    logger_b = create_notebook_logger('trackA', 'stageB')
    logger_b.log_start()
    
    try:
        # Import and run
        sys.path.insert(0, str(LOCAL_EXTRACTION / 'trackA' / 'trackA_stageB'))
        import trackA_stageB
        
        # Run stage B
        result = trackA_stageB.main()
        
        # Log results
        logger_b.log_metrics({'exit_code': result})
        logger_b.log_end(success=(result == 0))
        logger_b.print_summary()
        
    except Exception as e:
        logger_b.log_end(success=False, error=str(e))
        print(f"Stage B failed: {e}")
else:
    print("Stage B skipped (set RUN_STAGE_B = True to run)")

---
## 5. Track B: FGTP Fusion

Train multi-scale fusion model with dual cross-attention.

In [ ]:
# Load Track B configuration
trackB_cfg = load_config('trackB', overrides=CONFIG_OVERRIDES)

print("Track B Configuration:")
print(f"  Token dim: {trackB_cfg.get('model.projector.out_dim')}")
print(f"  Fusion layers: {trackB_cfg.get('model.fusion.layers')}")
print(f"  Fusion heads: {trackB_cfg.get('model.fusion.heads')}")
print(f"  Head hidden: {trackB_cfg.get('model.head.hidden')}")
print(f"  Learning rate: {trackB_cfg.get('training.lr')}")
print(f"  Epochs: {trackB_cfg.get('training.epochs')}")
print(f"  Demo mode: {trackB_cfg.get('demo.enabled')}")

## 5.1 Video Backbone Selection

Choose between ResNet18 (baseline) and VideoMAE (ego-only) backbones.
VideoMAE requires pretraining on ego video before Track B training.

In [ ]:
# Video backbone configuration
# Choose: "resnet18" (baseline, ImageNet pretrained) or "videomae_ego" (ego pretrained)

VIDEO_BACKBONE = "resnet18"  # Change to "videomae_ego" for ego-only variant

# Load appropriate config preset based on backbone
if VIDEO_BACKBONE == "videomae_ego":
    trackB_variant_cfg = load_config('trackB_videomae_ego', overrides=CONFIG_OVERRIDES)
    print("🎬 Using VideoMAE Ego-Only Backbone")
    print("  - Pretrained on egocentric video (in-domain)")
    print("  - Projector: 768 → 256 (VideoMAE embedding)")
    print("  - Requires: videomae_ego_encoder.pt weights")
else:
    trackB_variant_cfg = load_config('trackB_resnet18_baseline', overrides=CONFIG_OVERRIDES)
    print("🖼️ Using ResNet18 Baseline Backbone")
    print("  - Pretrained on ImageNet (exocentric transfer)")
    print("  - Projector: 512 → 256 (ResNet features)")
    print("  - Ready to train immediately")

print(f"\nVideo backbone: {trackB_variant_cfg.get('model.tokenizer.video_backbone')}")
print(f"Projector in_dim: {trackB_variant_cfg.get('model.projector.in_dim')}")

## 5.2 VideoMAE Pretraining (Optional)

Self-supervised pretraining on Ego4D clips. Only required if using `VIDEO_BACKBONE = "videomae_ego"`.

**Note:** This step is computationally expensive (~2-4 hours on GPU). Skip if using ResNet18 baseline.

In [ ]:
# VideoMAE Pretraining Configuration
RUN_VIDEOMAE_PRETRAIN = False  # Set to True to run pretraining

VIDEOMAE_PRETRAIN_CONFIG = {
    'num_frames': 16,        # Frames per clip
    'img_size': 224,         # Frame resolution
    'mask_ratio': 0.75,      # 75% of patches masked
    'epochs': 100,           # Pretraining epochs (reduce for demo)
    'batch_size': 8,         # Reduce if OOM
    'learning_rate': 1.5e-4,
    'warmup_epochs': 10,
    'save_freq': 10,         # Save checkpoint every N epochs
}

# Check for existing weights
videomae_weights = paths.local_extraction / 'runs' / 'VideoMAE' / 'videomae_ego_encoder.pt'
if videomae_weights.exists():
    print(f"✓ VideoMAE weights found: {videomae_weights}")
    print("  Pretraining can be skipped")
else:
    print("✗ VideoMAE weights not found")
    print(f"  Expected at: {videomae_weights}")
    if VIDEO_BACKBONE == "videomae_ego":
        print("  ⚠️ Run pretraining first!")

print(f"\nPretraining config: {VIDEOMAE_PRETRAIN_CONFIG}")

In [ ]:
# Run VideoMAE Pretraining
if RUN_VIDEOMAE_PRETRAIN and VIDEO_BACKBONE == "videomae_ego":
    print("Running VideoMAE Pretraining...")
    print("This will take several hours on GPU. Progress will be logged.\n")
    
    # Create run logger
    logger_vmae = create_notebook_logger('trackB', 'videomae_pretrain')
    logger_vmae.log_config(VIDEOMAE_PRETRAIN_CONFIG)
    logger_vmae.log_start()
    
    try:
        # Add videomae module to path
        sys.path.insert(0, str(LOCAL_EXTRACTION / 'trackB' / 'videomae'))
        
        from trackB.videomae.videomae_pretrain import PretrainConfig, pretrain_videomae
        
        # Create config
        pretrain_cfg = PretrainConfig(
            num_frames=VIDEOMAE_PRETRAIN_CONFIG['num_frames'],
            img_size=VIDEOMAE_PRETRAIN_CONFIG['img_size'],
            mask_ratio=VIDEOMAE_PRETRAIN_CONFIG['mask_ratio'],
            epochs=VIDEOMAE_PRETRAIN_CONFIG['epochs'],
            batch_size=VIDEOMAE_PRETRAIN_CONFIG['batch_size'],
            learning_rate=VIDEOMAE_PRETRAIN_CONFIG['learning_rate'],
            warmup_epochs=VIDEOMAE_PRETRAIN_CONFIG['warmup_epochs'],
            save_freq=VIDEOMAE_PRETRAIN_CONFIG['save_freq'],
        )
        
        # Run pretraining
        final_loss = pretrain_videomae(pretrain_cfg)
        
        logger_vmae.log_metrics({'final_loss': final_loss})
        logger_vmae.log_artifacts([str(videomae_weights)])
        logger_vmae.log_end(success=True)
        logger_vmae.print_summary()
        
        print(f"\n✓ VideoMAE pretraining complete!")
        print(f"  Final loss: {final_loss:.4f}")
        print(f"  Weights saved to: {videomae_weights}")
        
    except Exception as e:
        logger_vmae.log_end(success=False, error=str(e))
        print(f"VideoMAE pretraining failed: {e}")
        raise

elif RUN_VIDEOMAE_PRETRAIN and VIDEO_BACKBONE != "videomae_ego":
    print("⚠️ Pretraining skipped - VIDEO_BACKBONE is not 'videomae_ego'")
else:
    print("VideoMAE pretraining skipped (set RUN_VIDEOMAE_PRETRAIN = True to run)")

In [ ]:
# Run Track B Training (uses selected VIDEO_BACKBONE)
RUN_TRACKB_TRAIN = False  # Set to True to run

if RUN_TRACKB_TRAIN:
    print(f"Running Track B Training with {VIDEO_BACKBONE} backbone...")
    
    # Validate VideoMAE weights if using that backbone
    if VIDEO_BACKBONE == "videomae_ego" and not videomae_weights.exists():
        raise FileNotFoundError(
            f"VideoMAE weights not found at {videomae_weights}\n"
            "Run VideoMAE pretraining first (set RUN_VIDEOMAE_PRETRAIN = True)"
        )
    
    # Create run logger  
    logger_tb = create_notebook_logger('trackB')
    logger_tb.log_config({
        'video_backbone': VIDEO_BACKBONE,
        **CONFIG_OVERRIDES
    })
    logger_tb.log_start()
    
    try:
        # Import training module
        import trackB_train_loader
        
        # Override the config name based on backbone
        config_name = 'trackB_videomae_ego' if VIDEO_BACKBONE == 'videomae_ego' else 'trackB_resnet18_baseline'
        
        # Run training with appropriate config
        # The CLI args are parsed at module load, so we set them via environment or direct call
        import subprocess
        cmd = [
            sys.executable, '-m', 'trackB.trackB_train_loader',
            '--config', config_name,
        ]
        if CONFIG_OVERRIDES.get('demo', {}).get('enabled', True):
            cmd.append('--demo')
        
        print(f"Running: {' '.join(cmd)}")
        result = subprocess.run(cmd, cwd=str(LOCAL_EXTRACTION))
        
        logger_tb.log_metrics({'exit_code': result.returncode})
        logger_tb.log_end(success=(result.returncode == 0))
        logger_tb.print_summary()
        
    except Exception as e:
        logger_tb.log_end(success=False, error=str(e))
        print(f"Track B training failed: {e}")
        raise
else:
    print(f"Track B training skipped (set RUN_TRACKB_TRAIN = True to run)")
    print(f"  Selected backbone: {VIDEO_BACKBONE}")

In [ ]:
# Run Track B Evaluation
RUN_TRACKB_EVAL = False  # Set to True to run

if RUN_TRACKB_EVAL:
    print("Running Track B Evaluation...")
    
    try:
        import trackB_eval
        
        # Find latest checkpoint
        ckpt_dir = paths.local_extraction / 'runs' / 'Track_B' / 'checkpoints'
        ckpt_files = list(ckpt_dir.glob('trackB_best*.pt')) if ckpt_dir.exists() else []
        
        if ckpt_files:
            ckpt_path = sorted(ckpt_files)[-1]
            print(f"Using checkpoint: {ckpt_path}")
            
            # Run evaluation (has internal run logging)
            metrics = trackB_eval.evaluate(trackB_eval.EvalConfig(checkpoint_path=ckpt_path))
            print(f"Evaluation complete: mAP={metrics.get('mAP', 'N/A')}")
        else:
            print("No checkpoint found - run Track B training first")
            
    except Exception as e:
        print(f"Track B evaluation failed: {e}")
else:
    print("Track B evaluation skipped (set RUN_TRACKB_EVAL = True to run)")

## 5.4 Ablation: VideoMAE vs ResNet18

Compare results from both backbone variants.

In [ ]:
# Generate ablation comparison table
RUN_ABLATION = False  # Set to True after training both variants

if RUN_ABLATION:
    try:
        from trackB.summarize_ablation import find_runs, generate_table
        
        runs_dir = paths.local_extraction / 'runs' / 'Track_B'
        runs = find_runs(runs_dir)
        
        if runs:
            print(f"Found {len(runs)} Track B runs\n")
            
            # Generate comparison table
            table = generate_table(runs)
            print(table)
            
            # Save to file
            output_path = paths.local_extraction.parent / 'results' / 'ablation_videomae_vs_clip.md'
            output_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_path, 'w') as f:
                f.write(table)
            print(f"\n✓ Saved to: {output_path}")
        else:
            print("No Track B runs found with metrics.")
            print("Train both ResNet18 and VideoMAE variants first.")
            
    except Exception as e:
        print(f"Ablation comparison failed: {e}")
else:
    print("Ablation comparison skipped (set RUN_ABLATION = True to run)")
    print("Tip: Train with both VIDEO_BACKBONE options first")

---
## 6. Track C: RGTP Pruning

Apply token pruning with importance scoring.

In [ ]:
# Load Track C configuration
trackC_cfg = load_config('trackC', overrides=CONFIG_OVERRIDES)

print("Track C Configuration:")
print(f"  RGTP enabled: {trackC_cfg.get('rgtp.enabled')}")
print(f"  Pruning rate: {trackC_cfg.get('rgtp.rate')}")
print(f"  Min keep: {trackC_cfg.get('rgtp.min_keep')}")
print(f"  Temporal decay: {trackC_cfg.get('rgtp.temporal_decay')}")
print(f"  Measure latency: {trackC_cfg.get('instrumentation.measure_latency')}")

In [ ]:
# Run Track C Pruning Evaluation (uses selected VIDEO_BACKBONE)
RUN_TRACKC = False  # Set to True to run

if RUN_TRACKC:
    # Find Track B checkpoint matching the backbone
    ckpt_pattern = f'trackB_*_{VIDEO_BACKBONE}*.pt' if VIDEO_BACKBONE == 'videomae_ego' else 'trackB_best_*.pt'
    ckpt_path = paths.find_checkpoint(ckpt_pattern)
    
    if not ckpt_path:
        # Fallback to any checkpoint
        ckpt_path = paths.find_checkpoint('trackB_best_*.pt') or paths.find_checkpoint('trackB_final_*.pt')
    
    if ckpt_path:
        print(f"Applying RGTP pruning to: {ckpt_path}")
        print(f"Video backbone: {VIDEO_BACKBONE}")
        
        # Run Track C with backbone setting
        import subprocess
        cmd = [
            sys.executable, '-m', 'trackC.trackC_pruning',
            '--video_backbone', VIDEO_BACKBONE,
            '--checkpoint', str(ckpt_path),
        ]
        
        # Add rate if configured
        rate = trackC_cfg.get('rgtp.rate', 0.1)
        cmd.extend(['--rgtp_rate', str(rate)])
        
        print(f"Running: {' '.join(cmd)}")
        result = subprocess.run(cmd, cwd=str(LOCAL_EXTRACTION))
        
        if result.returncode == 0:
            print("✓ Track C evaluation complete")
        else:
            print(f"✗ Track C failed with exit code {result.returncode}")
    else:
        print("No checkpoint found - run Track B first")
else:
    print("Track C pruning skipped (set RUN_TRACKC = True to run)")
    print(f"  Will use backbone: {VIDEO_BACKBONE}")

---
## 7. Smoke Tests

Validate pipeline with quick smoke tests.

In [ ]:
# Run smoke tests (includes VideoMAE tests)
RUN_SMOKE_TESTS = True  # Set to False to skip

if RUN_SMOKE_TESTS:
    print("Running Smoke Tests...\n")
    
    all_passed = True
    
    # Track A
    try:
        from tests.smoke_test_trackA import run_smoke_tests as run_trackA_tests
        success, results = run_trackA_tests()
        print(f"Track A: {'✓ PASSED' if success else '✗ FAILED'}")
        all_passed = all_passed and success
    except ImportError as e:
        print(f"Track A: ⚠ Skipped ({e})")
    
    # Track B (includes VideoMAE tests)
    try:
        from tests.smoke_test_trackB import run_smoke_tests as run_trackB_tests
        success, results = run_trackB_tests()
        print(f"Track B: {'✓ PASSED' if success else '✗ FAILED'}")
        
        # Show VideoMAE test details
        for test in results.get('tests', []):
            if 'videomae' in test.get('name', '').lower():
                status = '✓' if test.get('passed', False) else '✗'
                print(f"  {status} {test['name']}: {test.get('message', '')}")
        
        all_passed = all_passed and success
    except ImportError as e:
        print(f"Track B: ⚠ Skipped ({e})")
    
    # Track C (includes VideoMAE backbone support tests)
    try:
        from tests.smoke_test_trackC import run_smoke_tests as run_trackC_tests
        success, results = run_trackC_tests()
        print(f"Track C: {'✓ PASSED' if success else '✗ FAILED'}")
        all_passed = all_passed and success
    except ImportError as e:
        print(f"Track C: ⚠ Skipped ({e})")
    
    # Track B unit tests (from trackB_tests.py)
    try:
        from trackB.trackB_tests import (
            test_videomae_model_shapes,
            test_videomae_masking,
            test_tokenizer_config_video_backbone,
        )
        print("\nVideoMAE Unit Tests:")
        test_videomae_model_shapes()
        print("  ✓ VideoMAE model shapes")
        test_videomae_masking()
        print("  ✓ VideoMAE masking")
        test_tokenizer_config_video_backbone()
        print("  ✓ TokenizerConfig video_backbone")
    except Exception as e:
        print(f"  ⚠ VideoMAE unit tests skipped: {e}")
    
    print(f"\n{'='*40}")
    print(f"Overall: {'✓ ALL PASSED' if all_passed else '✗ SOME FAILED'}")
else:
    print("Smoke tests skipped")

---
## 8. Results & Checkpoints

In [ ]:
# List available runs
runs_dir = paths.local_extraction / 'runs'

if runs_dir.exists():
    print("Available runs:")
    for track_dir in sorted(runs_dir.iterdir()):
        if track_dir.is_dir():
            print(f"\n{track_dir.name}/")
            for run_dir in sorted(track_dir.iterdir()):
                if run_dir.is_dir():
                    # Check for summary.json
                    summary = run_dir / 'summary.json'
                    has_summary = '✓' if summary.exists() else ' '
                    print(f"  {has_summary} {run_dir.name}")
else:
    print("No runs directory found yet")

In [ ]:
# Find and display latest results
latest_run = paths.latest_run(track='Track_A')

if latest_run:
    print(f"Latest Track A run: {latest_run}")
    
    summary_path = latest_run / 'summary.json'
    if summary_path.exists():
        import json
        with open(summary_path) as f:
            summary = json.load(f)
        print("\nSummary:")
        for key, value in summary.items():
            if not isinstance(value, (dict, list)):
                print(f"  {key}: {value}")
else:
    print("No Track A runs found")

---
## 9. Custom Configuration Examples

Examples of how to customize configs programmatically.

In [ ]:
# Example: Full training configuration
FULL_TRAINING_OVERRIDES = {
    'demo': {
        'enabled': False,
    },
    'training': {
        'epochs': 50,
        'batch_size': 16,
        'lr': 1e-4,
    },
    'logging': {
        'save_checkpoints': True,
        'checkpoint_freq': 5,
    },
}

# Load with full training settings
# full_cfg = load_config('trackB', overrides=FULL_TRAINING_OVERRIDES)
print("Full training config ready (uncomment to use)")

In [ ]:
# Example: Rate sweep for Track C
RATE_SWEEP_OVERRIDES = {
    'rate_sweep': {
        'enabled': True,
        'rates': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
    },
    'evaluation': {
        'save_per_rate': True,
        'plot_curve': True,
    },
}

# rate_sweep_cfg = load_config('trackC', overrides=RATE_SWEEP_OVERRIDES)
print("Rate sweep config ready (uncomment to use)")

---
## 10. Cleanup & Export

In [ ]:
# Export current config to YAML
EXPORT_CONFIG = False

if EXPORT_CONFIG:
    import yaml
    
    export_path = paths.local_extraction / 'configs' / 'custom_run.yaml'
    
    # Merge all overrides
    merged_cfg = load_config('trackB', overrides=CONFIG_OVERRIDES)
    
    with open(export_path, 'w') as f:
        yaml.dump(merged_cfg.raw, f, default_flow_style=False)
    
    print(f"Config exported to: {export_path}")
else:
    print("Config export skipped")

In [ ]:
# Session summary
print("="*50)
print("Session Summary")
print("="*50)
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Device: {device}")
print(f"Repo: {REPO_ROOT}")
print(f"\nAvailable configs:")
for cfg_file in (paths.local_extraction / 'configs').glob('*.yaml'):
    print(f"  - {cfg_file.name}")
print("\nDone!")